In [0]:
storage_account_name = "starchiveprojhexaware01"
storage_account_key = "3xsbiRSJLSvNVCyvdUcV2dZMpzQ/Dzydhiij0QaZd+crOO3f/OFyTgiM9ZpxTJNChaBzmlP6YIMY+AStIsvX3A=="

spark.conf.set(
  f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
  storage_account_key
)

In [0]:
df = spark.read.format("delta") \
    .load(
        f"abfss://silver@{storage_account_name}.dfs.core.windows.net/ecommerce"
    )

display(df)

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
548326,85078,SCANDINAVIAN 3 HEARTS NAPKIN RING,24,null,0.19,15424,United Kingdom
548340,22025,RING OF ROSES BIRTHDAY CARD,12,null,0.42,13426,United Kingdom
548353,22683,FRENCH BLUE METAL DOOR SIGN 8,15,null,1.25,15980,United Kingdom
548358,22196,SMALL HEART MEASURING SPOONS,12,null,0.85,15832,United Kingdom
548492,21975,PACK OF 60 DINOSAUR CAKE CASES,10,null,0.55,17841,United Kingdom
548512,21714,CITRONELLA CANDLE GARDEN POT,6,null,1.25,14410,United Kingdom
548544,22219,LOVEBIRD HANGING DECORATION WHITE,10,null,0.85,14525,United Kingdom
548553,23231,WRAP DOILEY DESIGN,25,null,0.42,12523,France
548555,22138,BAKING SET 9 PIECE RETROSPOT,3,null,4.95,13758,United Kingdom
548666,23177,TREASURE ISLAND BOOK BOX,3,null,2.25,13124,United Kingdom


In [0]:
from pyspark.sql.functions import *

df = df.withColumn(
    "Revenue",
    col("Quantity") * col("UnitPrice")
)

display(df)

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
548326,85078,SCANDINAVIAN 3 HEARTS NAPKIN RING,24,null,0.19,15424,United Kingdom,4.5600000000000005
548340,22025,RING OF ROSES BIRTHDAY CARD,12,null,0.42,13426,United Kingdom,5.04
548353,22683,FRENCH BLUE METAL DOOR SIGN 8,15,null,1.25,15980,United Kingdom,18.75
548358,22196,SMALL HEART MEASURING SPOONS,12,null,0.85,15832,United Kingdom,10.2
548492,21975,PACK OF 60 DINOSAUR CAKE CASES,10,null,0.55,17841,United Kingdom,5.5
548512,21714,CITRONELLA CANDLE GARDEN POT,6,null,1.25,14410,United Kingdom,7.5
548544,22219,LOVEBIRD HANGING DECORATION WHITE,10,null,0.85,14525,United Kingdom,8.5
548553,23231,WRAP DOILEY DESIGN,25,null,0.42,12523,France,10.5
548555,22138,BAKING SET 9 PIECE RETROSPOT,3,null,4.95,13758,United Kingdom,14.850000000000001
548666,23177,TREASURE ISLAND BOOK BOX,3,null,2.25,13124,United Kingdom,6.75


In [0]:
gold_df = df.groupBy("Country") \
    .agg(
        round(sum("Revenue"), 2).alias("TotalRevenue")
    )

display(gold_df)

Country,TotalRevenue
Sweden,38367.83
Singapore,21279.29
Germany,228678.4
RSA,1002.31
France,208934.31
Greece,4760.52
European Community,1300.25
Belgium,41196.34
Finland,22546.08
Malta,2725.59


In [0]:
gold_df.write.format("delta") \
    .mode("overwrite") \
    .save(
        f"abfss://gold@{storage_account_name}.dfs.core.windows.net/revenue_by_country"
    )

In [0]:
final_df = spark.read.format("delta") \
    .load(
        f"abfss://gold@{storage_account_name}.dfs.core.windows.net/revenue_by_country"
    )

display(final_df)

Country,TotalRevenue
Sweden,38367.83
Singapore,21279.29
Germany,228678.4
RSA,1002.31
France,208934.31
Greece,4760.52
European Community,1300.25
Belgium,41196.34
Finland,22546.08
Malta,2725.59
